# 03 · Recording

**What you will see**

- `robot.run(policy, record=dir)` turns one run into one episode of a **LeRobot v3 dataset**, written by LeRobot's own writer.
- Three episodes of the SO-101 with a camera, then a **second run resumes** the same directory.
- The dataset read back with LeRobot's `LeRobotDataset`, here and in a fresh process, and its frames played as video.
- `meta/units.json`: which units the numbers are in (radians, or LeRobot's units under a LeRobot checkpoint).
- `hf://owner/name` targets: parsed into LeRobot's cache, **nothing uploaded** unless you pass `push=True` (we don't).

Recording is a keyword on the same loop, not a separate tool.

In [1]:
import json, math, subprocess, sys, tempfile
from pathlib import Path
import numpy as np
import _nb
from strands_robots import Policy, Robot
_nb.show(_nb.drawing("v03_one_loop"))

## Three episodes, one camera

An attached camera becomes a video feature of the dataset (`observation.images.front`). `instruction=` becomes the episode's task. Recording needs an integer rate; we use 20 Hz.

In [2]:
work = Path(tempfile.mkdtemp(prefix="nb03-"))
target = work / "so101-waves"

def so101():
    a = Robot("so101")
    a.sim.camera("front", position=[0.45, -0.45, 0.35], target=[0, 0, 0.12], size=(160, 120), attach=a)
    return a

def wave(sign):
    return lambda o: {"shoulder_pan": sign * 0.9 * math.sin(3 * o.t), "elbow_flex": 0.6 * math.sin(2 * o.t)}

arm = so101()
rows = []
for ep, (sign, task) in enumerate([(1, "wave right"), (-1, "wave left"), (1, "wave right")]):
    r = arm.run(wave(sign), hz=20, steps=30, record=target, instruction=task)
    rows.append({"episode": ep, "task": task, "steps": r.steps, "stopped_by": r.stopped_by, "recorded": r.recorded.replace(str(work), "<work>")})
arm.sim.close()
info = json.loads((target / "meta" / "info.json").read_text())
print("codebase", info["codebase_version"], "| robot_type", info["robot_type"], "| episodes", info["total_episodes"], "| frames", info["total_frames"], "| fps", info["fps"])
_nb.table(rows)

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

codebase v3.0 | robot_type so101_follower | episodes 3 | frames 90 | fps 20


| episode | task | steps | stopped_by | recorded |
|---|---|---|---|---|
| 0 | wave right | 30 | steps | /private<work>/so101-waves |
| 1 | wave left | 30 | steps | /private<work>/so101-waves |
| 2 | wave right | 30 | steps | /private<work>/so101-waves |

## A second run resumes

A new robot, a new run, the same directory: the episode index continues and the earlier episodes are untouched. A run that would change the dataset's shape (here a different rate) is refused before step one.

In [3]:
arm = so101()
r = arm.run(wave(-1), hz=20, steps=20, record=target, instruction="wave left")
info = json.loads((target / "meta" / "info.json").read_text())
print("after resume: episodes", info["total_episodes"], "| frames", info["total_frames"])
try:
    arm.run(wave(1), hz=10, steps=5, record=target)
except ValueError as e:
    print("refused:", str(e).splitlines()[0].replace(str(work), "<work>"))
arm.sim.close()

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

after resume: episodes 4 | frames 110
refused: <work>/so101-waves: the dataset there is 20 fps; this run records at 10


## Read it back with LeRobot

`LeRobotDataset` is LeRobot's own reader; nothing strands-specific is needed to load the data. The same reload was also tested in a **fresh process** (so nothing cached here can make it look right); we do that too.

In [4]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset("local/so101-waves", root=target)
first = ds[0]
print("episodes", ds.num_episodes, "| frames", ds.num_frames, "| fps", ds.fps)
print("state/action names:", ds.meta.features["observation.state"]["names"])
print("ds[0]:", {k: tuple(v.shape) for k, v in first.items() if hasattr(v, "shape") and v.ndim > 0}, "| task:", first["task"])
print("frames per episode:", np.bincount([int(ds[i]["episode_index"]) for i in range(ds.num_frames)]).tolist())

fresh = subprocess.run([sys.executable, "-c",
    "import sys; from lerobot.datasets.lerobot_dataset import LeRobotDataset as D; d = D('local/x', root=sys.argv[1]); "
    "print(d.num_episodes, d.num_frames, d[d.num_frames - 1]['task'])", str(target)],
    capture_output=True, text=True, timeout=300)
print("fresh process:", fresh.stdout.strip().splitlines()[-1], "(episodes, frames, last task)")

episodes 4 | frames 110 | fps 20
state/action names: ['shoulder_pan.pos', 'shoulder_lift.pos', 'elbow_flex.pos', 'wrist_flex.pos', 'wrist_roll.pos', 'gripper.pos']
ds[0]: {'observation.state': (6,), 'action': (6,), 'observation.images.front': (3, 120, 160)} | task: wave right


frames per episode: [30, 30, 30, 20]


fresh process: 4 110 wave left (episodes, frames, last task)


## The frames in the dataset

Decoded from the dataset's own MP4s (not from the simulator), all four episodes back to back.

In [5]:
cam = "observation.images.front"
frames = [(ds[i][cam].permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8) for i in range(ds.num_frames)]
print(len(frames), "frames", frames[0].shape, "| pixel std", round(float(frames[0].std()), 1))
_nb.video(frames, fps=ds.fps, width=480)

110 frames (120, 160, 3) | pixel std 74.7


## `meta/units.json`: what the numbers mean

Written next to LeRobot's own `meta/`. A function policy works in radians, so the dataset says `rad` (no conversion to describe).

In [6]:
print(json.dumps(json.loads((target / "meta" / "units.json").read_text()), indent=1)[:900])

{
 "units": "rad",
 "joints": {}
}


Under a LeRobot checkpoint, the dataset stores what the policy saw and did, in **its** units (SO-101 default: −100…100, gripper 0…100), and says so. Radians into that dataset would be refused.

In [7]:
sys.path.insert(0, str(_nb.ROOT / "tests" / "policy"))
from tiny_act import save_tiny_act
ckpt = save_tiny_act(work / "tiny-act", 6, camera="front")
arm = Robot("so101")
arm.sim.camera("front", size=(64, 48), attach=arm)
act_target = work / "so101-act"
r = arm.run(Policy(str(ckpt), device="cpu"), hz=30, steps=15, record=act_target)
said = json.loads((act_target / "meta" / "units.json").read_text())
print("units:", said["units"], "| shoulder_pan.pos", said["joints"]["shoulder_pan.pos"], "| gripper.pos range", said["joints"]["gripper.pos"]["range"])
try:
    arm.run(lambda o: o.joints.positions, hz=30, steps=3, record=act_target)
except ValueError as e:
    print("refused:", str(e).splitlines()[0].replace(str(work), "<work>"))
arm.sim.close()

Loading weights from local directory


Map:   0%|          | 0/15 [00:00<?, ? examples/s]

units: lerobot | shoulder_pan.pos {'range': [-100.0, 100.0], 'rad': [-1.9198621771937616, 1.9198621771937634]} | gripper.pos range [0.0, 100.0]
refused: <work>/so101-act: the dataset there is stored in units 'lerobot'; this run records 'rad'


## Hub targets: `hf://owner/name`

Only the `hf://` prefix means the Hub; any other string is a directory. A Hub target records into LeRobot's local cache. **Nothing is uploaded without `push=True`.** Here the cache points at a temp dir, the Hub is offline, and any upload attempt would be counted instead of sent.

In [8]:
import os
from strands_robots.recording import is_repo_id, repo_id
from strands_robots.recording.target import locate
for t in ["hf://owner/so101-waves", "owner/so101-waves", "data/duck"]:
    print(f"{t!r:32} hub={is_repo_id(t)!s:5} ->", locate(t)[0])
for bad in ["hf://nope", "hf://a/b/c"]:
    try:
        repo_id(bad)
    except ValueError as e:
        print("refused:", e)

import lerobot.utils.constants as constants
os.environ["HF_HUB_OFFLINE"] = "1"
constants.HF_LEROBOT_HOME = work / "lerobot-cache"
pushed = []
LeRobotDataset.push_to_hub = lambda self, *a, **k: pushed.append(self.repo_id)   # count, never send

arm = so101()
r = arm.run(wave(1), hz=20, steps=10, record="hf://owner/so101-waves")   # push=False is the default
print("recorded into:", r.recorded.replace(str(work), "<work>"), "| uploads:", len(pushed))
try:
    arm.run(wave(1), hz=20, steps=2, record=work / "plain-dir", push=True)
except ValueError as e:
    print("refused:", str(e).splitlines()[0].replace(str(work), "<work>"))
arm.sim.close()
assert pushed == []

'hf://owner/so101-waves'         hub=True  -> owner/so101-waves
'owner/so101-waves'              hub=False -> local/so101-waves
'data/duck'                      hub=False -> local/duck
refused: a Hub target is hf://owner/name, got 'hf://nope'
refused: a Hub target is hf://owner/name, got 'hf://a/b/c'


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

recorded into: <work>/lerobot-cache/owner/so101-waves | uploads: 0
refused: push=True uploads to the Hub and needs record='hf://owner/name', got PosixPath('<work>/plain-dir')


## Where this sits in v1

Recording lives in `strands_robots/recording/` and is reached only through `run(..., record=, push=)`; `RunResult.recorded` says where the episode went. It depends only on types, description and units, and LeRobot is imported only when a run records. A run that faults records nothing; `stop()` keeps its frames. The same keyword works on a simulated arm and a real one, so sim episodes and hardware episodes land in the same LeRobot format, ready for training the policies from **02 · Policies**.